# 04 - Customer Segmentation
**Approach:** scale numerics, one-hot categoricals, PCA (90% variance), K-Means. Churn is **not** used to build clusters, only to profile them afterwards. That makes the churn separation a genuine test of whether the segments are meaningful.

In [1]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import eda, segment as S
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')
df = eda.load()
pipe, X = S.make_features(df)
print('raw features:', pipe.named_steps['prep'].transform(df).shape[1], '-> PCA components:', X.shape[1])

raw features: 21 -> PCA components: 11


**Why scale before K-Means?** K-Means uses Euclidean distance, so a feature measured in dollars (range ~20-120) would swamp a 0/1 flag. Standardising numerics puts them on a comparable footing with the one-hot columns.

**Why PCA?** One-hot encoding creates correlated columns (e.g. `InternetService=No` implies no streaming). PCA removes that redundancy and gives K-Means a cleaner space.

## 1. Choosing k

In [2]:
scan = S.scan_k(X)
stab = {k: S.stability(X, k) for k in [3, 4, 5, 6]}
scan['stability_ARI'] = scan.k.map(lambda k: round(stab[k][0], 3) if k in stab else np.nan)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(scan.k, scan.inertia, 'o-'); ax[0].set(title='Elbow (inertia)', xlabel='k')
ax[1].plot(scan.k, scan.silhouette, 'o-', c='C1'); ax[1].set(title='Silhouette (higher = better)', xlabel='k')
ax[2].plot(scan.k, scan.davies_bouldin, 'o-', c='C2'); ax[2].set(title='Davies-Bouldin (lower = better)', xlabel='k')
for a in ax: a.axvline(4, ls='--', c='grey')
plt.tight_layout(); plt.savefig('../reports/figures/07_choose_k.png', dpi=150); plt.show()
scan.round(3)

,k,inertia,silhouette,davies_bouldin,stability_ARI
0,2,30295.960,0.247,1.567,NaN
1,3,24348.372,0.238,1.474,0.902
2,4,21295.623,0.233,1.502,0.994
3,5,19316.364,0.217,1.588,0.915
4,6,17754.073,0.218,1.579,0.991
5,7,16884.246,0.198,1.552,NaN
6,8,16067.391,0.186,1.719,NaN
7,9,15569.195,0.175,1.814,NaN
8,10,15145.155,0.171,1.899,NaN


**Honest reading of the metrics.**
- Silhouette is low everywhere (~0.23 at k=2-4). Customer data is a continuum with fuzzy boundaries, not well-separated blobs, so I treat the clusters as *useful partitions*, not natural species.
- k=2 has the top silhouette but only splits 'committed' from 'not committed', which is already visible in the contract column.
- **k=4** has a near-minimal Davies-Bouldin score, a visible elbow, and by far the best stability among the candidates (**ARI 0.99** on 80% subsamples, vs 0.90 at k=3 and 0.92 at k=5), so the same four groups reappear regardless of the sample.
- k=4 also gives groups a retention team can act on differently. I chose k=4.

## 2. Fit and sanity-check against hierarchical clustering

In [3]:
km = S.fit_kmeans(X, 4)
ward_labels, ari = S.ward_agreement(X, 4, km.labels_)
print(f'Adjusted Rand Index, K-Means vs Ward: {ari:.2f}')
pd.crosstab(km.labels_, ward_labels, rownames=['kmeans'], colnames=['ward'])

Adjusted Rand Index, K-Means vs Ward: 0.50


ward,0,1,2,3
kmeans,,,,
0,10,11,1521,0
1,96,892,0,1049
2,1696,3,0,258
3,908,597,2,0


**ARI = 0.50: moderate agreement.** Ward recovers the same broad structure (e.g. the budget phone-only cluster maps 99% onto a single Ward cluster) but draws the boundaries between the middle groups differently. This is consistent with the low silhouette: the extremes (very risky / very safe) are robust, the middle is fuzzy. I keep K-Means because it is far more stable under resampling.

## 3. Profile the segments

In [4]:
prof = S.profile(df, km.labels_)
names = S.name_segments(prof)
prof.insert(0, 'segment', prof.index.map(names))
prof = prof.sort_values('churn_rate_pct', ascending=False)
prof.T

cluster,2,3,1,0
segment,"At-risk fiber, month-to-month","Mid-tier DSL, month-to-month",Loyal premium,Budget phone-only
customers,1957,1507,2037,1542
pct_base,27.8,21.4,28.9,21.9
churn_rate_pct,56.6,27.6,11.4,7.3
share_of_churners_pct,59.3,22.3,12.4,6.0
avg_tenure,17.7,17.6,58.6,30.7
avg_monthly,85.5,50.1,88.6,21.3
avg_protective,0.79,1.22,2.72,0.0
top_contract,Month-to-month,Month-to-month,Two year,Two year
pct_top_contract,96,78,48,42


## 4. Visuals

In [5]:
df['cluster'] = km.labels_; df['segment'] = df.cluster.map(names)
order = prof.segment.tolist()
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
# PCA scatter
rng = np.random.default_rng(0)
for s in order:
    m = (df.segment == s).values
    ax[0].scatter(X[m, 0], X[m, 1], s=6, alpha=.35, label=s)
ax[0].set(title='Segments in PCA space (first two components)', xlabel='PC1', ylabel='PC2')
ax[0].legend(markerscale=3, fontsize=8)
# churn rate bars
ax[1].barh(order, prof.churn_rate_pct, color='#E4572E')
for i, (v, sh) in enumerate(zip(prof.churn_rate_pct, prof.share_of_churners_pct)):
    ax[1].text(v + 1, i, f'{v:.0f}% churn | {sh:.0f}% of all churners', va='center', fontsize=9)
ax[1].axvline(df.Churn.mean()*100, ls='--', c='grey'); ax[1].invert_yaxis()
ax[1].set(title='Churn rate by segment (churn NOT used to cluster)', xlim=(0, 100), xlabel='Churn rate (%)')
plt.tight_layout(); plt.savefig('../reports/figures/08_segments.png', dpi=150); plt.show()

## 5. Business value: revenue at risk by segment

In [6]:
g = df.groupby('segment')
rev = pd.DataFrame({'customers': g.size(), 'churn_rate_pct': (g.Churn.mean()*100).round(1),
    'total_MRR': g.MonthlyCharges.sum().round(0),
    'MRR_lost_to_churn': (df.MonthlyCharges * df.Churn).groupby(df.segment).sum().round(0)})
rev['pct_of_lost_MRR'] = (rev.MRR_lost_to_churn / rev.MRR_lost_to_churn.sum() * 100).round(1)
rev = rev.sort_values('MRR_lost_to_churn', ascending=False); rev

,customers,churn_rate_pct,total_MRR,MRR_lost_to_churn,pct_of_lost_MRR
segment,,,,,
"At-risk fiber, month-to-month",1957,56.6,167261.0,94788.0,68.1
Loyal premium,2037,11.4,180443.0,22569.0,16.2
"Mid-tier DSL, month-to-month",1507,27.6,75563.0,19473.0,14.0
Budget phone-only,1542,7.3,32850.0,2302.0,1.7


## 6. Do the clusters rediscover the Phase 3 hand-built segment?

In [7]:
a = (df.Contract=='Month-to-month') & (df.tenure<=12) & (df.InternetService=='Fiber optic')
pd.crosstab(a.map({True:'EDA segment A (new, m2m, fiber)', False:'everyone else'}), df.segment, normalize='index').mul(100).round(1)

segment,"At-risk fiber, month-to-month",Budget phone-only,Loyal premium,"Mid-tier DSL, month-to-month"
row_0,,,,
"EDA segment A (new, m2m, fiber)",100.0,0.0,0.0,0.0
everyone else,17.0,25.2,33.2,24.6


**Result:** 100% of the hand-built Phase 3 segment A (new, month-to-month, fiber) lands inside the 'At-risk fiber, month-to-month' cluster, so unsupervised clustering independently recovered it and extended it to 1,957 customers (segment A had 916).

**Revenue:** this one segment accounts for **68% of all monthly revenue lost to churn** despite being 28% of customers.

## 7. Segment names and recommended plays
(Numbers from the profile table above.)

- **At-risk fiber, month-to-month:** highest churn and the largest share of churners *and* of lost revenue. They pay ~$85/month, are on short contracts, rarely have protective add-ons, and 63% pay by e-check. **Play:** offer a discounted 12-month fiber contract bundled with security/tech-support, and nudge e-check payers to auto-pay.
- **Mid-tier DSL, month-to-month:** moderate churn (~28%), lower bills. **Play:** low-cost upgrade path to one-year terms; do not over-spend on discounts.
- **Loyal premium:** high tenure, high spend, ~11% churn, two-year contracts, many add-ons. **Play:** protect, not discount. Loyalty perks, early-renewal outreach before contracts lapse.
- **Budget phone-only:** lowest churn and lowest value (~$21/mo). **Play:** no retention spend.

**Limitations.** K-Means treats one-hot columns as numeric, so mixed-type structure is only approximated (K-Prototypes or Gower distance with k-medoids are alternatives). Silhouette is low, so boundaries between middle segments are soft.

In [8]:
out = df[['customerID', 'cluster', 'segment']]
out.to_csv('../data/segments.csv', index=False)
out.segment.value_counts()

segment
Loyal premium                    2037
At-risk fiber, month-to-month    1957
Budget phone-only                1542
Mid-tier DSL, month-to-month     1507
Name: count, dtype: int64